In [1]:
import torch
torch.manual_seed(123)

In [2]:
# “Your journey starts with one step”
inputs = torch.tensor(
[[0.43, 0.15, 0.89], # Your (x^1)
[0.55, 0.87, 0.66], # journey (x^2)
[0.57, 0.85, 0.64], # starts (x^3)
[0.22, 0.58, 0.33], # with (x^4)
[0.77, 0.25, 0.10], # one (x^5)
[0.05, 0.80, 0.55]] # step (x^6)
)

In [3]:

#TODO: our goal for now is only to compute the one context vector, z(2),
# 
x_2 = inputs[1] # second input element x^(2) as the query
d_in = inputs.shape[1] # d=3 which is the size of our (each) input embedding vector
d_out = 2

In [4]:

W_query = torch.nn.Parameter(torch.rand(d_in, d_out), requires_grad=False)
W_key = torch.nn.Parameter(torch.rand(d_in, d_out), requires_grad=False)
W_value = torch.nn.Parameter(torch.rand(d_in, d_out), requires_grad=False)

In [5]:
query_2 = x_2 @ W_query
key_2 = x_2 @ W_key
value_2 = x_2 @ W_value


In [6]:
print(query_2)

tensor([0.4306, 1.4551])



$$
Q = XW_Q
$$
$$
K = XW_K
$$
$$
V = XW_V
$$

Note that $W_Q , W_K, W_V$ are learned parameters that are optimized during training

While :
$AttentionWeights=softmax(\frac{ QK^T}{\sqrt{d_k}})$

$AttentionScores = QK^T$

attention weights are dynamic, context-specific values.

In [ ]:

#* We will need key and value vectors for all input elements even though our goal is calculate context vector Z^2

keys = inputs @ W_key
values = inputs @ W_value
print("keys.shape:", keys.shape)
print("values.shape:", values.shape)

keys.shape: torch.Size([6, 2])
values.shape: torch.Size([6, 2])


**we successfully projected the six input tokens from a three-dimensional onto a two-dimensional embedding space**


In [ ]:

# TODO: Next step: compute the attention scores

attn_weights = query_2 @ keys.T
d_k = keys.shape[-1] # keys.shape(6,2)->keys.shape[-1]-> 2
attn_score_2 = torch.softmax(input=(attn_weights/d_k**0.5),dim=-1)
attn_score_2

# All attention weights for given query

tensor([0.1500, 0.2264, 0.2199, 0.1311, 0.0906, 0.1820])

**We just calculated Scaled Dot-Product Attention**

In [ ]:

# TODO: the final step is to compute the context vectors
#* multiplying each value vector with its respective attention weight and then summing them to obtain the context vector

context_vector_2 = attn_score_2 @ values
context_vector_2

tensor([0.3061, 0.8210])

### Implementing self-attention class

In [27]:
import torch.nn as nn
torch.manual_seed(123)

class SelfAttention_v1(nn.Module):
    def __init__(self,d_in,d_out):
        super().__init__()
        self.W_query = nn.Parameter(torch.rand(d_in, d_out))
        self.W_key = nn.Parameter(torch.rand(d_in, d_out))
        self.W_value = nn.Parameter(torch.rand(d_in, d_out))
    
    def forward(self,X):
        keys = X @ self.W_key
        queries = X @ self.W_query
        values = X @ self.W_value
        attn_scores = queries @ keys.T
        d_k = keys.shape[-1]
        attn_weights = torch.softmax(input=(attn_scores/(d_k**0.5)),dim=-1)
        context_vec = attn_weights @ values
        return context_vec



In [28]:
# “Your journey starts with one step”
inputs = torch.tensor(
[[0.43, 0.15, 0.89], # Your (x^1)
[0.55, 0.87, 0.66], # journey (x^2)
[0.57, 0.85, 0.64], # starts (x^3)
[0.22, 0.58, 0.33], # with (x^4)
[0.77, 0.25, 0.10], # one (x^5)
[0.05, 0.80, 0.55]] # step (x^6)
)

In [29]:
sa_v1 = SelfAttention_v1(d_in=inputs.shape[-1],d_out=2)
print(sa_v1(inputs))

tensor([[0.2996, 0.8053],
        [0.3061, 0.8210],
        [0.3058, 0.8203],
        [0.2948, 0.7939],
        [0.2927, 0.7891],
        [0.2990, 0.8040]], grad_fn=<MmBackward0>)


<div>
<img src = "../helpful_imgs/self_attention_withQKV.png",width=600>
</div>



$Attention(Q,K,V) = softmax(\frac{QK^T}{\sqrt{d_k}})V$
$$
Q = XW_Q
$$
$$
K = XW_K
$$
$$
V = XW_V
$$


$AttentionWeights=softmax(\frac{ QK^T}{\sqrt{d_k}})$ #scaled

$ContextVector = AttentionWeights . V $



In [12]:
# nn.Linear has an optimized weight initialization scheme, contributing to more stable and
# effective model training
import torch.nn as nn
import torch
torch.manual_seed(123)

class SelfAttention_v2(nn.Module):
    def __init__(self,d_in,d_out,qkv_bias=False):
        super().__init__()
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
    
    def forward(self,X):
        keys = self.W_key(X)
        queries = self.W_query(X)
        values = self.W_value(X)
        attn_scores = queries @ keys.T
        d_k = keys.shape[-1]
        attn_weights = torch.softmax(input=(attn_scores/(d_k**0.5)),dim=-1)
        context_vec = attn_weights @ values
        return context_vec



In [13]:
# “Your journey starts with one step”
inputs = torch.tensor(
[[0.43, 0.15, 0.89], # Your (x^1)
[0.55, 0.87, 0.66], # journey (x^2)
[0.57, 0.85, 0.64], # starts (x^3)
[0.22, 0.58, 0.33], # with (x^4)
[0.77, 0.25, 0.10], # one (x^5)
[0.05, 0.80, 0.55]] # step (x^6)
)
d_in,d_out = inputs.shape[1],2
sa_v2 = SelfAttention_v2(d_in, d_out)
print(sa_v2(inputs))

tensor([[-0.5337, -0.1051],
        [-0.5323, -0.1080],
        [-0.5323, -0.1079],
        [-0.5297, -0.1076],
        [-0.5311, -0.1066],
        [-0.5299, -0.1081]], grad_fn=<MmBackward0>)


In [19]:

# TODO: Exercise 3.1

sa_v2.W_key.weight , sa_v2.W_query.weight , sa_v2.W_value.weight


(Parameter containing:
 tensor([[-0.4196, -0.4590, -0.3648],
         [ 0.2615, -0.2133,  0.2161]], requires_grad=True),
 Parameter containing:
 tensor([[-0.2354,  0.0191, -0.2867],
         [ 0.2177, -0.4919,  0.4232]], requires_grad=True),
 Parameter containing:
 tensor([[-0.4900, -0.3503, -0.2120],
         [-0.1135, -0.4404,  0.3780]], requires_grad=True))

In [33]:
sa_v1.W_key = torch.nn.Parameter(data=sa_v2.W_key.weight.T)
sa_v1.W_query = torch.nn.Parameter(data=sa_v2.W_query.weight.T)
sa_v1.W_value = torch.nn.Parameter(data=sa_v2.W_value.weight.T)
print(sa_v1(inputs))

tensor([[-0.5337, -0.1051],
        [-0.5323, -0.1080],
        [-0.5323, -0.1079],
        [-0.5297, -0.1076],
        [-0.5311, -0.1066],
        [-0.5299, -0.1081]], grad_fn=<MmBackward0>)


**Proves that both implementation are same**